<a href="https://colab.research.google.com/github/yelsebety/Yassin-Flyrank/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Model vs. Baseline (Lane 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yelsebety/Yassin-Flyrank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Question shape:** *"Which pages should a reviewer look at first?"* with an observed (proxy) label, "declining" (`trend_direction == "down"`). That is a ranking problem built from a probability, so I fit classifiers and judge them by **precision@K** (and ROC AUC for the overall ordering), against my Week-4 rule baseline on the **same pages and the same splits**.

**Methods, simple to complex:**
- **Logistic Regression**: readable coefficients, my first try.
- **Decision Tree (depth 3)**: small enough to print and read.
- **Random Forest** and **Gradient Boosting**: stronger on interactions, harder to read. Each has to earn its place.
- Not clustering: I have a target, so there is nothing to discover.

**A rule fixed before looking at results:** among the models, I pick the **simplest one whose development-fold AUC is within 0.005 of the best**. So complexity alone does not win.

**Data and features (same slice as the Week-4 baseline):** the 12,023 starter-CSV pages with 500+ impressions and average position 1–20. Features (all measured before and independent of the label): `log_impressions`, `avg_position`, `log_ctr`, `content_age_days`, `days_since_last_update`, `days_with_impressions`, `log_word_count`, `log_search_volume`, `competition`, `cpc`, `engagement_rate`, plus `content_type` and `main_intent`. Blanks get a median fill **plus a "was missing" flag** (a blind `fillna(0)` would hide content type in the data). Never used: `trend_direction`, `trend_pct`, any `*_last_30d` / `*_prev_30d` column (the label is made from them), the IDs, and product flags.

**Honest limit:** the 90-day totals and the proxy label come from the same window, so this compares methods on a proxy; it is not a forecast of future declines.

In [1]:
import os, sys, subprocess, json, warnings
import numpy as np
import pandas as pd
import sklearn
warnings.filterwarnings("ignore")

from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance

SEED = 0
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/yelsebety/Yassin-Flyrank.git"
REPO_DIR = "Yassin-Flyrank"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)

raw = pd.read_csv("data/raw/content_refresh_anonymized.csv")
raw["declined"] = (raw["trend_direction"] == "down").astype(int)      # label: evaluation/target only

# Same slice as the Week-4 baseline
g = raw[(raw["impressions_90d"] >= 500) & (raw["avg_position"] > 0) & (raw["avg_position"] <= 20)
        & (raw["content_age_days"] >= 90)].copy().reset_index(drop=True)

# Safe features only (log for heavy-tailed counts; percentages stay as given)
g["log_impressions"]   = np.log1p(g["impressions_90d"])
g["log_ctr"]           = np.log1p(g["ctr"])
g["log_word_count"]    = np.log1p(g["word_count"])
g["log_search_volume"] = np.log1p(g["search_volume"])
NUM = ["log_impressions", "avg_position", "log_ctr", "content_age_days", "days_since_last_update",
       "days_with_impressions", "log_word_count", "log_search_volume", "competition", "cpc", "engagement_rate"]
CAT = ["content_type", "main_intent"]
FEATURES = NUM + CAT

BANNED = ["trend_direction", "trend_pct", "declined", "impressions_last_30d", "impressions_prev_30d",
          "clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d", "content_id", "client_id"]
assert not set(FEATURES) & set(BANNED), "a banned column is a feature"

print(f"scikit-learn {sklearn.__version__}, seed {SEED}")
print(f"Pages: {len(g):,} | clients: {g['client_id'].nunique()} | base rate of the proxy label: {g['declined'].mean():.3f}")
print("Share missing per feature (kept as blanks + flags):")
print(g[FEATURES].isna().mean().round(3)[lambda s: s > 0].to_dict())

# ---------- helpers ----------
def precision_at_k(y, score, impressions, k):
    """Precision in the top-k. Ties in score are broken by impressions (more demand first)."""
    order = np.lexsort((-np.asarray(impressions), -np.asarray(score)))
    return float(np.asarray(y)[order][:k].mean())

def evaluate(y, score, impressions):
    return {"auc": roc_auc_score(y, score), "p@20": precision_at_k(y, score, impressions, 20),
            "p@50": precision_at_k(y, score, impressions, 50), "base_rate": float(np.mean(y))}

def baseline_scores(train, apply):
    """Week-4 rule: relative CTR shortfall vs. the median CTR of the same position band (medians from TRAIN only)."""
    cut = lambda d: pd.cut(d["avg_position"], [0, 3, 10, 20], labels=["a", "b", "c"])
    med = train.assign(b=cut(train)).groupby("b", observed=True)["ctr"].median()
    typical = cut(apply).map(med).astype(float)
    return ((typical - apply["ctr"]) / typical).clip(lower=0)

def preprocessor(scale):
    num = [("fill", SimpleImputer(strategy="median", add_indicator=True))] + ([("scale", StandardScaler())] if scale else [])
    cat = [("fill", SimpleImputer(strategy="constant", fill_value="unknown")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]
    return ColumnTransformer([("num", Pipeline(num), NUM), ("cat", Pipeline(cat), CAT)])

def make_models():
    # Fixed settings, NOT tuned on the evaluation folds
    return {
        "Logistic Regression":     Pipeline([("pre", preprocessor(True)),  ("m", LogisticRegression(C=1.0, max_iter=2000))]),
        "Decision Tree (depth 3)": Pipeline([("pre", preprocessor(False)), ("m", DecisionTreeClassifier(max_depth=3, min_samples_leaf=100, random_state=SEED))]),
        "Random Forest":           Pipeline([("pre", preprocessor(False)), ("m", RandomForestClassifier(n_estimators=300, min_samples_leaf=20, n_jobs=-1, random_state=SEED))]),
        "Gradient Boosting":       Pipeline([("pre", preprocessor(False)), ("m", HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=3, random_state=SEED))]),
    }
COMPLEXITY_ORDER = ["Logistic Regression", "Decision Tree (depth 3)", "Random Forest", "Gradient Boosting"]

scikit-learn 1.6.1, seed 0
Pages: 12,023 | clients: 28 | base rate of the proxy label: 0.599
Share missing per feature (kept as blanks + flags):
{'log_word_count': 0.305, 'log_search_volume': 0.019, 'competition': 0.019, 'cpc': 0.019, 'main_intent': 0.014}


## 2. Split design

**Grouped by client, never by page.** Pages from the same client share a site, topic mix and tracking setup, so a random page-level split would let the model "recognise" a client it already saw, and the score would look better than it should. A real user of this ranking meets *new* clients, so I hold out whole clients.

**Two layers:**
1. **A sealed test set:** about 20% of the clients (chosen once with a fixed seed). It is not used to choose a model. I look at it once, at the end.
2. **Development set (the other clients):** 5-fold **GroupKFold** on `client_id`. Every model and the baseline are scored on the same folds, so the comparison is fair. The baseline's "typical CTR" values are computed from each fold's training clients only.

**Why not time-aware:** the starter CSV is one 90-day snapshot with no dates, so there is no time axis to split on. That is a real limit (see section 4).

In [2]:
# Sealed-test rule, fixed from the DESIGN (not from any model result):
# about 20% of clients, and no single client may be more than 40% of the sealed pages.
def pick_split():
    tried = []
    for seed in range(50):
        tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed).split(g, groups=g["client_id"]))
        share = g.iloc[te]["client_id"].value_counts(normalize=True).iloc[0]
        tried.append((seed, round(float(share), 3)))
        if share <= 0.40:
            return seed, tr, te, tried
SPLIT_SEED, dev_idx, test_idx, tried = pick_split()
print(f"Split seeds tried (seed, biggest client's share of the sealed pages): {tried}")
print(f"-> using seed {SPLIT_SEED}. (Seed 0 was rejected because one client was most of the sealed pages, which would make the 'sealed test' a one-client test.)\n")

dev  = g.iloc[dev_idx].reset_index(drop=True)
test = g.iloc[test_idx].reset_index(drop=True)

assert set(dev["client_id"]).isdisjoint(set(test["client_id"])), "a client is in both dev and sealed test"
print(f"Development: {dev['client_id'].nunique()} clients, {len(dev):,} pages, base rate {dev['declined'].mean():.3f}")
print(f"Sealed test: {test['client_id'].nunique()} clients, {len(test):,} pages, base rate {test['declined'].mean():.3f}")

folds = list(GroupKFold(n_splits=5).split(dev, groups=dev["client_id"]))
for i, (tr, va) in enumerate(folds):
    assert set(dev.iloc[tr]["client_id"]).isdisjoint(set(dev.iloc[va]["client_id"]))
    print(f"  fold {i}: train {len(tr):,} pages / validate {len(va):,} pages ({dev.iloc[va]['client_id'].nunique()} unseen clients)")

Split seeds tried (seed, biggest client's share of the sealed pages): [(0, 0.779), (1, 0.474), (2, 0.837), (3, 0.431), (4, 0.556), (5, 0.645), (6, 0.877), (7, 0.794), (8, 0.726), (9, 0.468), (10, 0.607), (11, 0.654), (12, 0.866), (13, 0.965), (14, 0.895), (15, 0.812), (16, 0.665), (17, 0.54), (18, 0.925), (19, 0.386)]
-> using seed 19. (Seed 0 was rejected because one client was most of the sealed pages, which would make the 'sealed test' a one-client test.)

Development: 22 clients, 10,659 pages, base rate 0.597
Sealed test: 6 clients, 1,364 pages, base rate 0.609
  fold 0: train 6,264 pages / validate 4,395 pages (1 unseen clients)
  fold 1: train 9,095 pages / validate 1,564 pages (4 unseen clients)
  fold 2: train 9,084 pages / validate 1,575 pages (2 unseen clients)
  fold 3: train 9,096 pages / validate 1,563 pages (8 unseen clients)
  fold 4: train 9,097 pages / validate 1,562 pages (7 unseen clients)


## 3. Train + compare vs my baseline

**Step A (development folds):** the baseline rule and all four models, same folds, same metrics. **Step B (sealed test):** pick one model with my pre-set rule, refit on all development clients, and compare it to the baseline once on the held-out clients.

In [3]:
rows = []
for fold, (tr, va) in enumerate(folds):
    a, b = dev.iloc[tr], dev.iloc[va]
    rows.append({"model": "Baseline rule (Week 4)", "fold": fold,
                 **evaluate(b["declined"], baseline_scores(a, b), b["impressions_90d"])})
    for name, model in make_models().items():
        model.fit(a[FEATURES], a["declined"])
        p = model.predict_proba(b[FEATURES])[:, 1]
        rows.append({"model": name, "fold": fold, **evaluate(b["declined"], p, b["impressions_90d"])})
cv = pd.DataFrame(rows)

base = cv[cv["model"] == "Baseline rule (Week 4)"].set_index("fold")
summary = []
for name, d in cv.groupby("model", sort=False):
    d = d.set_index("fold")
    summary.append({"model": name,
                    "AUC (mean)": d["auc"].mean(), "AUC (sd)": d["auc"].std(),
                    "P@20 (mean)": d["p@20"].mean(), "P@50 (mean)": d["p@50"].mean(), "P@50 (sd)": d["p@50"].std(),
                    "folds won vs baseline (P@50)": int((d["p@50"] > base["p@50"]).sum()) if name != "Baseline rule (Week 4)" else "-",
                    "folds won vs baseline (AUC)":  int((d["auc"] > base["auc"]).sum())  if name != "Baseline rule (Week 4)" else "-"})
dev_table = pd.DataFrame(summary).round(3)
print(f"Development folds (5 grouped folds). Base rate (mean of folds): {cv['base_rate'].mean():.3f}")
display(dev_table)

Development folds (5 grouped folds). Base rate (mean of folds): 0.615


,model,AUC (mean),AUC (sd),P@20 (mean),P@50 (mean),P@50 (sd),folds won vs baseline (P@50),folds won vs baseline (AUC)
0,Baseline rule (Week 4),0.593,0.041,0.83,0.820,0.116,-,-
1,Logistic Regression,0.605,0.054,0.80,0.808,0.052,2,3
2,Decision Tree (depth 3),0.582,0.020,0.77,0.708,0.208,3,3
3,Random Forest,0.611,0.022,0.90,0.852,0.059,3,3
4,Gradient Boosting,0.600,0.035,0.86,0.816,0.105,3,3


In [4]:
# Pre-set rule: simplest model whose dev AUC is within 0.005 of the best model's.
model_auc = dev_table[dev_table["model"] != "Baseline rule (Week 4)"].set_index("model")["AUC (mean)"]
best_auc = model_auc.max()
chosen = next(m for m in COMPLEXITY_ORDER if model_auc[m] >= best_auc - 0.005)
print(f"Best dev AUC: {best_auc:.3f} -> chosen by the pre-set rule: {chosen} (AUC {model_auc[chosen]:.3f})")

# Sealed test: refit on ALL development clients, evaluate once on the held-out clients.
final = {}
test_rows = [{"model": "Baseline rule (Week 4)",
              **evaluate(test["declined"], baseline_scores(dev, test), test["impressions_90d"])}]
for name, model in make_models().items():
    model.fit(dev[FEATURES], dev["declined"])
    final[name] = model
    p = model.predict_proba(test[FEATURES])[:, 1]
    test_rows.append({"model": name + ("  <- chosen" if name == chosen else ""),
                      **evaluate(test["declined"], p, test["impressions_90d"])})
test_table = pd.DataFrame(test_rows).round(3)
print(f"\nSealed test: {test['client_id'].nunique()} unseen clients, {len(test):,} pages")
display(test_table)

Best dev AUC: 0.611 -> chosen by the pre-set rule: Random Forest (AUC 0.611)

Sealed test: 6 unseen clients, 1,364 pages


,model,auc,p@20,p@50,base_rate
0,Baseline rule (Week 4),0.582,0.75,0.82,0.609
1,Logistic Regression,0.673,0.90,0.88,0.609
2,Decision Tree (depth 3),0.574,0.85,0.72,0.609
3,Random Forest <- chosen,0.661,0.95,0.90,0.609
4,Gradient Boosting,0.677,0.85,0.84,0.609


**What the two tables say (read together):**

- **Development folds (5 grouped folds):** the Week-4 baseline scores AUC 0.593 and precision@50 0.820. The models land at AUC 0.582–0.611 and precision@50 0.708–0.852, and each beats the baseline in only about 3 of the 5 folds (the depth-3 tree is the weakest). The fold-to-fold spread (P@50 standard deviation 0.05–0.21) is larger than the gaps between methods. So on development clients **I cannot call any model clearly better than my rule at the top of the list.**
- **Which model my pre-set rule picked:** Random Forest (AUC 0.611). Logistic Regression was 0.006 behind, just outside the 0.005 band.
- **Sealed test (6 unseen clients, 1,364 pages):** the chosen Random Forest gets AUC 0.661, precision@20 0.95 and precision@50 0.90, against 0.582, 0.75 and 0.82 for the baseline. Logistic Regression (0.673) and Gradient Boosting (0.677) also beat the baseline's AUC. **This is directional evidence that a learned score orders pages better than the single CTR rule**, mainly in the overall ordering (AUC).
- **Why I only call it directional:** the sealed set is small. In a top 50, a gap of 0.08 is four pages. The development folds do not show the same size of gain. Folds also differ a lot: fold 0 is a single client holding 4,395 pages.
- **Complexity is not earning much:** Logistic Regression is within 0.012 AUC of the Random Forest on the sealed clients, and the depth-3 tree does worse than the baseline. A readable logistic model would be a reasonable pick for the capstone.
- **How the sealed clients were chosen:** by a rule about the design (about 20% of clients, and no client above 40% of the sealed pages), not by any result. Seeds 0–18 broke that rule (one client was most of the pages), so I used seed 19. I did see results from the first, rejected split before making that change, so it is fair to say that I looked.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on?* The checks below use the chosen model on the sealed test clients: permutation importance (shuffle one feature and see how much AUC drops), the model's readable form, and a look at the wrong picks near the top of the list.

In [5]:
model = final[chosen]
y_test = test["declined"].to_numpy()

# 1) Permutation importance on the sealed clients (drop in AUC when one feature is shuffled)
pi = permutation_importance(model, test[FEATURES], y_test, scoring="roc_auc", n_repeats=10, random_state=SEED)
imp = (pd.DataFrame({"feature": FEATURES, "auc_drop": pi.importances_mean, "sd": pi.importances_std})
         .sort_values("auc_drop", ascending=False).round(4).reset_index(drop=True))
print("Permutation importance (sealed test clients):")
display(imp.head(8))

# 2) Readable form: the depth-3 tree printed as rules
tree_pipe = final["Decision Tree (depth 3)"]
names = list(tree_pipe.named_steps["pre"].get_feature_names_out())
print("Depth-3 decision tree, as rules:")
print(export_text(tree_pipe.named_steps["m"], feature_names=names))

# 3) The chosen model's strongest coefficients (only if it is the logistic regression)
if chosen == "Logistic Regression":
    coefs = pd.Series(model.named_steps["m"].coef_[0], index=model.named_steps["pre"].get_feature_names_out())
    print("Logistic regression: largest standardized coefficients (positive = more likely 'declining'):")
    display(coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(8).round(3).to_frame("coef"))

Permutation importance (sealed test clients):


,feature,auc_drop,sd
0,days_with_impressions,0.0665,0.0036
1,log_ctr,0.0563,0.0101
2,log_impressions,0.0438,0.0088
3,log_search_volume,0.0146,0.0038
4,log_word_count,0.0079,0.0066
5,avg_position,0.0074,0.0035
6,engagement_rate,0.0050,0.0032
7,main_intent,0.0008,0.0013


Depth-3 decision tree, as rules:
|--- num__content_age_days <= 312.50
|   |--- num__log_ctr <= 0.14
|   |   |--- num__log_impressions <= 10.03
|   |   |   |--- class: 1
|   |   |--- num__log_impressions >  10.03
|   |   |   |--- class: 1
|   |--- num__log_ctr >  0.14
|   |   |--- num__log_impressions <= 9.72
|   |   |   |--- class: 1
|   |   |--- num__log_impressions >  9.72
|   |   |   |--- class: 0
|--- num__content_age_days >  312.50
|   |--- num__days_with_impressions <= 87.50
|   |   |--- num__avg_position <= 8.75
|   |   |   |--- class: 1
|   |   |--- num__avg_position >  8.75
|   |   |   |--- class: 1
|   |--- num__days_with_impressions >  87.50
|   |   |--- num__log_ctr <= 0.34
|   |   |   |--- class: 0
|   |   |--- num__log_ctr >  0.34
|   |   |   |--- class: 0



In [6]:
# Error analysis: the model's top 50 on the sealed clients
test["p_declining"] = model.predict_proba(test[FEATURES])[:, 1]
order = np.lexsort((-test["impressions_90d"].to_numpy(), -test["p_declining"].to_numpy()))
top50 = test.iloc[order[:50]].copy()
top50["right"] = top50["declined"] == 1
print(f"Top 50: {int(top50['right'].sum())} right, {int((~top50['right']).sum())} wrong "
      f"(precision {top50['right'].mean():.2f}; base rate {test['declined'].mean():.2f}).")

compare_cols = ["avg_position", "log_ctr", "days_with_impressions", "content_age_days", "log_impressions"]
print("\nAverage of key features: right picks vs. wrong picks vs. all sealed-test pages")
display(pd.DataFrame({"right picks": top50[top50["right"]][compare_cols].mean(),
                      "wrong picks": top50[~top50["right"]][compare_cols].mean(),
                      "all test pages": test[compare_cols].mean()}).round(2))

print("\nWhere the top-50 picks come from (clients and content type):")
print(f"  distinct clients in the top 50: {top50['client_id'].nunique()} of {test['client_id'].nunique()}")
display(top50.groupby("content_type")["right"].agg(n="count", precision="mean").round(2))

print("\nThree concrete wrong picks (highest-ranked pages that were NOT declining):")
wrong3 = top50[~top50["right"]].head(3)[["content_id", "p_declining", "impressions_90d", "avg_position", "ctr",
                                           "days_with_impressions", "content_type", "trend_direction"]]
display(wrong3.round(3))

# Receipts: a small metrics file that is safe to commit
metrics = {"slice": "starter CSV gated slice (500+ impressions, position 1-20)", "seed": SEED,
           "sklearn": sklearn.__version__, "chosen_by_rule": chosen,
           "dev_cv": dev_table.to_dict(orient="records"), "sealed_test": test_table.to_dict(orient="records"),
           "caveat": "Proxy label from the same 90-day window as the features; grouped by client; sealed test is only a few clients, so it is noisy."}
os.makedirs("work/outputs", exist_ok=True)
with open("work/outputs/w05_model_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, default=str)
print("\nWrote work/outputs/w05_model_metrics.json")

Top 50: 45 right, 5 wrong (precision 0.90; base rate 0.61).

Average of key features: right picks vs. wrong picks vs. all sealed-test pages


,right picks,wrong picks,all test pages
avg_position,10.46,12.56,9.47
log_ctr,0.02,0.01,0.29
days_with_impressions,80.64,77.20,80.06
content_age_days,132.16,132.20,146.65
log_impressions,6.92,6.90,8.13



Where the top-50 picks come from (clients and content type):
  distinct clients in the top 50: 3 of 6


,n,precision
content_type,,
keyword article,50,0.9



Three concrete wrong picks (highest-ranked pages that were NOT declining):


,content_id,p_declining,impressions_90d,avg_position,ctr,days_with_impressions,content_type,trend_direction
1186,content_3bb23afaf34e,0.878,2910,15.4,0.03,88,keyword article,stable
121,content_96da95476e63,0.836,784,7.4,0.00,85,keyword article,stable
650,content_afb6250ef84b,0.836,521,18.3,0.00,69,keyword article,stable



Wrote work/outputs/w05_model_metrics.json


**Errors and interpretation**

- **What the model leans on:** `days_with_impressions` (AUC drops by 0.067 when shuffled), then `log_ctr` (0.056) and `log_impressions` (0.044). Everything else matters much less. CTR being near the top agrees with my Week-4 signal check. **A caution on `days_with_impressions`:** a page that only recently started getting impressions can look like it is "up" by construction, so the model may be partly learning "new to search" rather than real decline risk. I cannot separate those two with this data.
- **The readable form:** the printed depth-3 tree splits first on content age, then on CTR, impressions, and days with impressions. Young pages with low CTR end up on the "declining" side. The tree is the weakest scorer, so I read it as an illustration, not as the model.
- **Where the top 50 goes wrong:** 45 of the top 50 are right and 5 are wrong (precision 0.90 vs. a base rate of 0.61). The wrong picks sit a bit further down the results page (average position 12.6 vs. 10.5 for right picks), where a low CTR is more normal. The top 50 comes from only 3 of the 6 sealed clients, so the list is shaped by which clients are in the test.
- **Three concrete wrong picks** are all keyword articles marked "stable", with CTR near 0 (positions 15.4, 7.4 and 18.3; 2,910, 784 and 521 impressions). They are hard for two reasons. A low CTR at position 15–18 is common even for healthy pages. And "stable" in my proxy label just means impressions did not fall by more than 20%, so a page with a real 15% drop is on the wrong side of the line.
- **What this means:** the models look somewhat better than the rule at ordering pages, but the evidence is small and the label is a proxy from the same window as the features. These are observed, directional, decision-support results; they do not show that these pages will decline or that a refresh would help.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.